# Preprocessing — Teks bebas → kelompok makna (Laya + embedding sebagai dua juri)
**Kapan dipakai:** isian bebas yang **beda kata tapi sama makna**, misalnya pekerjaan `ojol`,
`ojek online`, `sopir`. Regex dan fuzzy tidak bisa menyamakan ini karena hurufnya berbeda.

**Langkah:** dua juri memilih satu kelompok untuk tiap **nilai unik**:
- **Laya** (`pretrained/laya`, multilingual): model keputusan "pilih satu opsi".
- **MiniLM** (`pretrained/minilm`): kemiripan embedding dengan deskripsi kelompok.

Kalau kedua juri **setuju**, usulan diterima. Kalau **tidak setuju**, nilai itu diperiksa manual
dan hasilnya ditulis di `CORRECTIONS`.

Hasil di data demo (36 nilai unik, 8 kelompok):
- Laya benar 22 dari 36, MiniLM juga 22 dari 36 (61%). Tidak cukup bagus untuk dipakai sendirian.
- Kedua juri setuju di 19 nilai, 18 di antaranya benar. Sisanya 17 nilai diperiksa manual.
- Yang setuju tapi salah: `notaris`. Jadi daftar "setuju" tetap dibaca sekilas.

Laya hanya dijalankan pada nilai unik (sekitar 0,5 detik per nilai di GPU), bukan semua baris.

**Data demo:** `pekerjaan_ayah` di `ukt_SINTETIS` (sama dengan `tabular/survei_kotor`).

In [1]:
import laya
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

## Konfigurasi

In [2]:
DATA_FOLDER = "../../data_latihan/tabular_extra/survei_kotor"   # GANTI saat lomba
TRAIN_CSV = f"{DATA_FOLDER}/ukt_SINTETIS_train.csv"
TEST_CSV = f"{DATA_FOLDER}/ukt_SINTETIS_test.csv"
LAYA_DIR = "../../pretrained/laya"
EMBED_DIR = "../../pretrained/minilm"
DEVICE = "cpu"
if torch.cuda.is_available():
    DEVICE = "cuda"

COLUMN = "pekerjaan_ayah"
PREFIX = "Pekerjaan ayah: "        # konteks yang ditempel di depan tiap nilai untuk Laya
QUESTION = "Pekerjaan ini termasuk kelompok mana?"
# Kelompok tujuan + deskripsi singkat. Deskripsi yang jelas = usulan yang lebih tepat.
GROUPS = {
    "tidak_bekerja": "tidak bekerja, belum bekerja, menganggur, kosong",
    "buruh": "buruh, kuli, pekerja kasar harian",
    "petani": "petani, pekebun, nelayan, peternak",
    "informal": "pedagang kecil, warung, ojek, ojol, usaha kecil",
    "karyawan": "karyawan swasta, sopir, satpam, guru honorer",
    "pns_rendah": "PNS golongan rendah (I, II), honorer pemerintah, perangkat desa",
    "pns": "PNS golongan III ke atas, TNI, Polri, guru PNS, pensiunan PNS",
    "profesional": "dokter, dosen, notaris, direktur, pengusaha besar",
}
# Diisi SETELAH membaca tabel "tidak setuju" (nilai unik -> kelompok)
CORRECTIONS = {"-": "tidak_bekerja", "tni": "pns", "polri": "pns", "pensiunan pns": "pns",
               "pns gol iii": "pns", "pns gol ii": "pns_rendah", "honorer pemda": "pns_rendah",
               "perangkat desa": "pns_rendah", "guru honorer": "karyawan", "satpam": "karyawan",
               "sopir": "karyawan", "wiraswasta warung": "informal", "ojol": "informal",
               "dokter": "profesional", "dosen": "profesional", "direktur": "profesional",
               "pengusaha": "profesional", "notaris": "profesional"}

## Baca data → nilai unik (kosong jadi "-", spasi dibuang, huruf kecil)

In [3]:
def clean_text(values):
    values = values.fillna("-")
    values = values.str.strip()
    return values.str.lower()


train = pd.read_csv(TRAIN_CSV)
test = pd.read_csv(TEST_CSV)
all_values = clean_text(pd.concat([train[COLUMN], test[COLUMN]]))
counts = all_values.value_counts()

draft = pd.DataFrame()
draft["value"] = counts.index
draft["count"] = counts.to_numpy()
print(len(draft), "nilai unik dari", len(all_values), "baris")
draft.head()

36 nilai unik dari 37975 baris


,value,count
0,karyawan swasta,2666
1,petani,1877
2,nelayan,1784
3,tidak bekerja,1596
4,buruh tani,1432


## Juri 1: Laya

In [4]:
judge = laya.load(LAYA_DIR, subfolder="multilingual", device=DEVICE)
question = {"group": {"type": "choice", "instructions": QUESTION, "criteria": GROUPS}}
prompts = []
for value in draft["value"]:
    prompts.append(PREFIX + value)
answers = judge.predict_batch(prompts, question)

laya_choices = []
laya_confidence = []
for answer in answers:
    group_answer = answer["answers"]["group"]
    laya_choices.append(group_answer["choice"])
    laya_confidence.append(max(group_answer["probabilities"].values()))
draft["laya"] = laya_choices
draft["laya_confidence"] = laya_confidence
draft.head()

,value,count,laya,laya_confidence
0,karyawan swasta,2666,karyawan,1.0000
1,petani,1877,petani,1.0000
2,nelayan,1784,petani,0.7172
3,tidak bekerja,1596,tidak_bekerja,0.7031
4,buruh tani,1432,buruh,0.9999


## Juri 2: embedding MiniLM (kemiripan kosinus dengan deskripsi kelompok)

In [5]:
encoder = SentenceTransformer(EMBED_DIR, device=DEVICE)
group_names = list(GROUPS.keys())
group_texts = []
for name, description in GROUPS.items():
    group_texts.append(name + ": " + description)
value_vectors = encoder.encode(list(draft["value"]), normalize_embeddings=True)
group_vectors = encoder.encode(group_texts, normalize_embeddings=True)
similarity = value_vectors @ group_vectors.T     # baris = nilai, kolom = kelompok

embedding_choices = []
for best_index in similarity.argmax(axis=1):
    embedding_choices.append(group_names[best_index])
draft["embedding"] = embedding_choices
draft["embedding_similarity"] = similarity.max(axis=1).round(2)
draft["agree"] = draft["laya"] == draft["embedding"]
draft.to_csv(f"peta_{COLUMN}_DRAF.csv", index=False)
print("setuju:", draft["agree"].sum(), "| perlu dicek:", (~draft["agree"]).sum())

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

setuju: 19 | perlu dicek: 17


## Periksa
Tabel pertama (tidak setuju) perlu diputuskan manual: tulis hasilnya di `CORRECTIONS`.
Tabel kedua (setuju) cukup dibaca sekilas. Yang salah juga ditulis di `CORRECTIONS`.

In [6]:
draft[~draft["agree"]]

,value,count,laya,laya_confidence,embedding,embedding_similarity,agree
5,wiraswasta warung,1424,karyawan,0.5700,buruh,0.26,False
7,sopir,1388,karyawan,0.9727,buruh,0.33,False
9,honorer pemda,1377,tidak_bekerja,0.6484,karyawan,0.50,False
10,satpam,1375,tidak_bekerja,0.5388,informal,0.37,False
12,guru honorer,1349,tidak_bekerja,0.4767,karyawan,0.47,False
14,perangkat desa,1329,tidak_bekerja,0.9082,petani,0.57,False
15,ojol,1311,tidak_bekerja,0.6153,buruh,0.44,False
16,pns gol ii,1310,pns_rendah,0.8674,tidak_bekerja,0.46,False
20,-,778,tidak_bekerja,0.4082,buruh,0.44,False
25,tni,625,tidak_bekerja,0.4167,buruh,0.38,False


In [7]:
draft[draft["agree"]].sort_values("laya_confidence")

,value,count,laya,laya_confidence,embedding,embedding_similarity,agree
28,guru pns,582,pns,0.4023,pns,0.51,True
8,pedagang kecil,1383,informal,0.6738,informal,0.72,True
19,belum bekerja,800,tidak_bekerja,0.6827,tidak_bekerja,0.42,True
3,tidak bekerja,1596,tidak_bekerja,0.7031,tidak_bekerja,0.34,True
2,nelayan,1784,petani,0.7172,petani,0.52,True
32,notaris,525,tidak_bekerja,0.7214,tidak_bekerja,0.42,True
13,ojek online,1329,informal,0.8628,informal,0.23,True
22,kuli angkut,724,buruh,0.9476,buruh,0.42,True
11,pedagang keliling,1374,informal,0.9688,informal,0.50,True
24,pns golongan iii,641,pns,0.9916,pns,0.61,True


## Terapkan peta

In [8]:
mapping = {}
for value, choice, agree in zip(draft["value"], draft["laya"], draft["agree"]):
    if agree:
        mapping[value] = choice
mapping.update(CORRECTIONS)

train_groups = clean_text(train[COLUMN]).map(mapping)
test_groups = clean_text(test[COLUMN]).map(mapping)
train_missing = train_groups.isna().mean()
test_missing = test_groups.isna().mean()
print("train tanpa kelompok:", round(train_missing, 3))
print("test tanpa kelompok:", round(test_missing, 3))
train_groups.value_counts(dropna=False)

train tanpa kelompok: 0.0
test tanpa kelompok: 0.0


pekerjaan_ayah
informal         5472
karyawan         5426
pns_rendah       4374
petani           4344
pns              2905
buruh            2861
tidak_bekerja    2524
profesional      2474
Name: count, dtype: int64

## Simpan peta
Pakai sebagai kolom kategori baru, misalnya di `tabular/survei_kotor`:
`train["kelompok_kerja"] = train_groups`, lalu tambahkan `"kelompok_kerja"` ke `CAT_COLS`.

In [9]:
mapping_table = pd.DataFrame()
mapping_table["nilai"] = list(mapping.keys())
mapping_table["kelompok"] = list(mapping.values())
mapping_table.to_csv(f"peta_{COLUMN}.csv", index=False)
mapping_table.head()

,nilai,kelompok
0,karyawan swasta,karyawan
1,petani,petani
2,nelayan,petani
3,tidak bekerja,tidak_bekerja
4,buruh tani,buruh
